# Task 2 – Retrieval-Augmented Generation (RAG) from scratch

**Documents:** two research papers (about 50 pages in total), both free on arXiv:
- *LoRA: Low-Rank Adaptation of Large Language Models* (Hu et al., 2021)
- *QLoRA: Efficient Finetuning of Quantized LLMs* (Dettmers et al., 2023)

(Bonus: reading them also helps with Task 1 and the 7B memory question.)

**No frameworks.** The pipeline is built by hand:
1. PDF → text, page by page
2. Split text into **chunks**
3. Turn every chunk into a vector with an **embedding model**
4. Store the vectors in a **FAISS** index
5. For a question: embed it → find the 5 most similar chunks (**retrieval**)
6. Give those chunks to an LLM → it writes the answer (**generation**)

**Colab:** Runtime → Change runtime type → **T4 GPU**.

## Step 1 – Install libraries
- `pypdf` reads PDFs
- `sentence-transformers` runs the embedding model
- `faiss-cpu` is a fast vector-search library from Meta
- `rank_bm25` is classic keyword search (used for comparison in Step 10)

In [ ]:
!pip install -q pypdf sentence-transformers faiss-cpu rank_bm25

import re, io, random, requests
import numpy as np, pandas as pd, faiss, torch
from pypdf import PdfReader

## Step 2 – Download the PDFs and extract text page by page
We keep the **paper name and page number** with every piece of text, so answers can cite their source.

Cleaning: PDFs break words across lines with a hyphen (`effi-\ncient`), so we join those back and collapse extra spaces.

In [ ]:
DOCS = {
    "LoRA":  "https://arxiv.org/pdf/2106.09685",
    "QLoRA": "https://arxiv.org/pdf/2305.14314",
}

def clean(text):
    text = re.sub(r"(\w)-\n(\w)", r"\1\2", text)   # join hyphenated line breaks
    return re.sub(r"\s+", " ", text).strip()

pages = []   # list of {"doc", "page", "text"}
for name, url in DOCS.items():
    pdf = PdfReader(io.BytesIO(requests.get(url, timeout=60).content))
    for i, p in enumerate(pdf.pages, 1):
        t = clean(p.extract_text() or "")
        if len(t) > 50:
            pages.append({"doc": name, "page": i, "text": t})
    print(f"{name}: {len(pdf.pages)} pages")

print("Total pages with text:", len(pages))
print("Total words:", sum(len(p["text"].split()) for p in pages))

## Step 3 – Tools for writing your questions

To score retrieval we need, for every question, the **correct source passage**. We record it as an **evidence phrase**: a short, exact piece of text (5–12 words) copied from the paper that contains the answer.

A retrieved chunk counts as a **hit** if it contains the evidence phrase. This works for **any** chunk size, so it's fair to compare chunk sizes.

- `show_random_passages()` prints random passages to read and get question ideas from
- `find("some phrase")` checks a phrase really exists in the papers and shows where

In [ ]:
def norm(s):
    return re.sub(r"\s+", " ", s.lower()).strip()

def find(phrase):
    hits = [(p["doc"], p["page"]) for p in pages if norm(phrase) in norm(p["text"])]
    print(f"'{phrase}' ->", hits if hits else "NOT FOUND – copy the text exactly from the passage")
    return hits

def show_random_passages(n=3, length=700, seed=None):
    rng = random.Random(seed)
    for p in rng.sample(pages, n):
        start = rng.randint(0, max(0, len(p["text"]) - length))
        print(f"--- {p['doc']} page {p['page']} ---")
        print(p["text"][start:start + length], "\n")

show_random_passages(seed=0)

## Step 4 – ✍️ YOUR 15 test questions

**The task says these must be questions you wrote yourself.** Three examples are below so you can see the format; write the other 12.

How to write a good one:
1. Run `show_random_passages()` (cell above) a few times and read a passage
2. Write a question **in your own words** – don't copy the sentence, rephrase it (that is what makes retrieval hard and realistic)
3. Copy a short **exact** phrase from the passage that contains the answer as `evidence`
4. Add a mix: some easy, some about numbers/tables, some that use different words from the paper

In [ ]:
QUESTIONS = [
    # (question, exact evidence phrase from the source passage)
    ('Compared with fully fine-tuning GPT-3, by how much does LoRA cut the number of trainable parameters?',
     '10,000 times'),
    ("What does the 'double quantization' trick in QLoRA quantize?",
     'quantizing the quantization constants'),
    ('What hardware does QLoRA need to fine-tune a 65-billion-parameter model?',
     'single 48GB GPU'),
    ("Why doesn't a LoRA model run slower than the original?",
     'no additional inference latency'),
    ('What does LoRA do with the original model weights?',
     'freezes the pre'),
    ('Which models did they test LoRA on?',
     'RoBERTa, DeBERTa, GPT-2, and GPT-3'),
    ('How much did training memory drop for GPT-3 with LoRA?',
     '1.2TB to 350GB'),
    ('How small does the saved checkpoint become with LoRA?',
     '350GB to 35MB'),
    ('What is B set to at the start of training?',
     'zero for B'),
    ('What did the QLoRA authors name their best chatbot models?',
     'Guanaco'),
    ('What new number format does QLoRA introduce?',
     '4-bit NormalFloat (NF4)'),
    ('Why does QLoRA use paged optimizers?',
     'paged optimizers to manage memory spikes'),
    ('How close is Guanaco to ChatGPT?',
     '99.3% of the performance level of ChatGPT'),
    ('How much GPU memory would normal 16-bit fine-tuning of LLaMA 65B need?',
     '780 GB of GPU memory'),
    ('How long did it take to train Guanaco?',
     '24 hours of finetuning on a single GPU'),
]
print(len(QUESTIONS), "questions")

In [ ]:
# check every evidence phrase exists in the papers (fix any that say NOT FOUND)
missing = [ev for q, ev in QUESTIONS if not find(ev)]
assert not missing, f"Fix these evidence phrases: {missing}"
assert len(QUESTIONS) >= 15, f"Only {len(QUESTIONS)} questions – the task needs 15"

## Step 5 – Chunking

We split each page into overlapping windows of words. We compare two sizes:

| | Chunk size | Overlap |
|---|---|---|
| **small** | 100 words (~1 paragraph) | 20 words |
| **large** | 300 words (~half a page) | 50 words |

**Overlap** means the end of one chunk is repeated at the start of the next, so a sentence cut at a boundary still appears whole in one chunk.

In [ ]:
def make_chunks(pages, size, overlap):
    chunks = []
    for p in pages:
        words = p["text"].split()
        step = size - overlap
        for start in range(0, max(1, len(words) - overlap), step):
            piece = " ".join(words[start:start + size])
            if len(piece.split()) >= 15:
                chunks.append({"doc": p["doc"], "page": p["page"], "text": piece})
    return chunks

SETTINGS = {"small (100 words)": (100, 20), "large (300 words)": (300, 50)}
chunk_sets = {name: make_chunks(pages, *cfg) for name, cfg in SETTINGS.items()}
for name, ch in chunk_sets.items():
    print(f"{name}: {len(ch)} chunks")

## Step 6 – Embedding model + FAISS index

**Embedding model:** `BAAI/bge-small-en-v1.5` – small (33M parameters) but strong at retrieval. It turns any text into a list of 384 numbers (a vector) so that texts with similar **meaning** get similar vectors.

BGE wants questions to start with a short instruction (`QUERY_PREFIX`); passages are embedded as they are.

**FAISS index:** `IndexFlatIP` = compare the question vector with **every** chunk vector using the inner product. Because we normalise vectors to length 1, inner product = **cosine similarity**. ("Flat" = exact search, no approximation – fine for a few hundred chunks.)

In [ ]:
from sentence_transformers import SentenceTransformer

device = "cuda" if torch.cuda.is_available() else "cpu"
embedder = SentenceTransformer("BAAI/bge-small-en-v1.5", device=device)
QUERY_PREFIX = "Represent this sentence for searching relevant passages: "

def embed(texts, is_query=False):
    if is_query:
        texts = [QUERY_PREFIX + t for t in texts]
    return embedder.encode(texts, normalize_embeddings=True, batch_size=64,
                           convert_to_numpy=True).astype("float32")

def build_index(chunks):
    vecs = embed([c["text"] for c in chunks])
    index = faiss.IndexFlatIP(vecs.shape[1])
    index.add(vecs)
    return index

indexes = {name: build_index(ch) for name, ch in chunk_sets.items()}
print({name: idx.ntotal for name, idx in indexes.items()})

## Step 7 – Retrieval and scoring

- `search(question, k=5)` returns the top-k chunks
- **Hit** = evidence phrase appears in a retrieved chunk
- **recall@k** = % of questions with at least one hit in the top k
- **MRR** (mean reciprocal rank) = on average how high the first correct chunk is (1.0 = always at rank 1, 0.5 = rank 2, …). Rewards putting the right chunk *first*.

In [ ]:
def search(question, setting, k=5):
    q = embed([question], is_query=True)
    scores, ids = indexes[setting].search(q, k)
    return [dict(chunk_sets[setting][i], score=float(s)) for s, i in zip(scores[0], ids[0])]

def first_hit_rank(retrieved, evidence):
    for rank, c in enumerate(retrieved, 1):
        if norm(evidence) in norm(c["text"]):
            return rank
    return None

def evaluate(retrieve_fn, k_max=20):
    rows = []
    for q, ev in QUESTIONS:
        r = first_hit_rank(retrieve_fn(q, k_max), ev)
        rows.append({"question": q, "rank": r})
    df = pd.DataFrame(rows)
    rank = df["rank"]
    summary = {"recall@1": (rank <= 1).mean(), "recall@5": (rank <= 5).mean(),
               "recall@10": (rank <= 10).mean(),
               "MRR@10": rank.where(rank <= 10).rdiv(1).fillna(0).mean()}
    return df, {k: round(v, 3) for k, v in summary.items()}

## Step 8 – Compare the two chunk sizes (main result for the report)

In [ ]:
per_question, summaries = {}, {}
for setting in SETTINGS:
    per_question[setting], summaries[setting] = evaluate(lambda q, k, s=setting: search(q, s, k))

results = pd.DataFrame(summaries).T
results

In [ ]:
# per-question view: rank of the first correct chunk (empty = not found in top 20)
detail = pd.DataFrame({"question": [q for q, _ in QUESTIONS]})
for setting, df in per_question.items():
    detail[setting] = df["rank"]
detail

**Why might one size win?** Things to think about when you write your answer:
- **Small chunks** = one focused idea each → the vector is "sharp", so a precise question matches it well. But a small chunk may cut off the context the answer needs.
- **Large chunks** = more context and the evidence is less likely to be split, but the vector is an *average* of many ideas, so a specific fact gets "diluted".
- Also: the large setting has fewer chunks, so even random guessing has a better chance of hitting in the top 5 – keep that in mind when comparing.

## Step 9 – Look at the failures

For each question that was **not** found in the top 5, see what was retrieved instead and where the right chunk actually ranked. Pick **2** for the report and explain **why** retrieval failed and **what would fix it**.

Typical reasons: the question uses different words than the paper (vocabulary mismatch) · the answer is in a table or formula that extracts badly · the evidence got split across two chunks · many passages talk about the same topic, so the wrong one looks just as similar.

In [ ]:
best = max(summaries, key=lambda s: summaries[s]["recall@5"])
print("Showing failures for:", best, "\n")
for (q, ev), r in zip(QUESTIONS, per_question[best]["rank"]):
    if pd.isna(r) or r > 5:
        print("QUESTION :", q)
        print("EVIDENCE :", ev, "| correct chunk rank:", ">20" if pd.isna(r) else int(r))
        for i, c in enumerate(search(q, best, 3), 1):
            print(f"  top{i} [{c['doc']} p{c['page']}, score {c['score']:.2f}]: {c['text'][:200]}...")
        print()

## Step 10 – (Bonus) BM25 keyword search and hybrid search

This gives you real numbers for the theory question *"what does BM25 add to embedding search?"*

- **BM25** scores chunks by **exact word overlap** with the question (rare words count more).
- **Hybrid** combines both rankings with **Reciprocal Rank Fusion (RRF)**: each chunk gets `1/(60 + rank)` from each method, and we add them up. Chunks ranked well by *either* method rise to the top.

In [ ]:
from rank_bm25 import BM25Okapi

tokenize = lambda s: re.findall(r"[a-z0-9]+", s.lower())
bm25 = {s: BM25Okapi([tokenize(c["text"]) for c in chunk_sets[s]]) for s in SETTINGS}

def bm25_search(question, setting, k=5):
    scores = bm25[setting].get_scores(tokenize(question))
    top = np.argsort(scores)[::-1][:k]
    return [dict(chunk_sets[setting][i], score=float(scores[i])) for i in top]

def hybrid_search(question, setting, k=5, pool=50, c=60):
    fused = {}
    for ranked in (search(question, setting, pool), bm25_search(question, setting, pool)):
        for rank, ch in enumerate(ranked, 1):
            key = (ch["doc"], ch["page"], ch["text"][:80])
            fused.setdefault(key, [0.0, ch])[0] += 1 / (c + rank)
    best_chunks = sorted(fused.values(), key=lambda x: -x[0])[:k]
    return [dict(ch, score=s) for s, ch in best_chunks]

rows = {}
for s in SETTINGS:
    rows[(s, "embedding")] = summaries[s]
    rows[(s, "BM25")]      = evaluate(lambda q, k, s=s: bm25_search(q, s, k))[1]
    rows[(s, "hybrid")]    = evaluate(lambda q, k, s=s: hybrid_search(q, s, k))[1]
method_comparison = pd.DataFrame(rows).T
method_comparison

## Step 11 – Generation: answer questions with an LLM

Now the "G" in RAG. We give `Qwen2.5-1.5B-Instruct` the top-5 chunks and tell it to answer **only from them** and cite the page. Retrieval decides *what the model sees* – if the right chunk is not retrieved, the model can't answer correctly.

In [ ]:
from transformers import AutoTokenizer, AutoModelForCausalLM

LLM = "Qwen/Qwen2.5-1.5B-Instruct"
llm_tok = AutoTokenizer.from_pretrained(LLM)
llm = AutoModelForCausalLM.from_pretrained(LLM).to(device, torch.float16)

def answer(question, setting=best, k=5):
    ctx = search(question, setting, k)
    context = "\n\n".join(f"[{c['doc']} p{c['page']}] {c['text']}" for c in ctx)
    msgs = [
        {"role": "system", "content": "Answer the question using ONLY the context. "
         "Cite the source like [LoRA p3]. If the context does not contain the answer, say you don't know."},
        {"role": "user", "content": f"Context:\n{context}\n\nQuestion: {question}"},
    ]
    prompt = llm_tok.apply_chat_template(msgs, tokenize=False, add_generation_prompt=True)
    ids = llm_tok(prompt, return_tensors="pt")["input_ids"].to(device)
    out = llm.generate(ids, max_new_tokens=150, do_sample=False)
    return llm_tok.decode(out[0, ids.shape[1]:], skip_special_tokens=True).strip()

for q, ev in QUESTIONS[:3]:
    print("Q:", q)
    print("A:", answer(q))
    print("(evidence:", ev, ")\n")

## Step 12 – Save results (download → upload to GitHub `task2_rag/`)

In [ ]:
results.to_csv("task2_chunk_size_results.csv")
detail.to_csv("task2_per_question_ranks.csv", index=False)
method_comparison.to_csv("task2_method_comparison.csv")
pd.DataFrame(QUESTIONS, columns=["question", "evidence"]).to_csv("task2_questions.csv", index=False)
try:
    from google.colab import files
    for f in ["task2_chunk_size_results.csv", "task2_per_question_ranks.csv",
              "task2_method_comparison.csv", "task2_questions.csv"]:
        files.download(f)
except ImportError:
    print("Saved – download from the Output panel")